In [ ]:
# =====================================
# STEP 1: UPLOAD DATA
# =====================================
from google.colab import files
files.upload()

# =====================================
# STEP 2: SETUP
# =====================================
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

# =====================================
# STEP 3: FIX FILE NAMES (AUTO CLEAN)
# =====================================
for file in os.listdir():
    f = file.lower()
    if 'customer' in f:
        os.rename(file, 'customers.csv')
    elif 'order' in f:
        os.rename(file, 'orders.csv')
    elif 'support' in f:
        os.rename(file, 'support_tickets.csv')
    elif 'web' in f:
        os.rename(file, 'web_events_snapshot.csv')
    elif 'intervention' in f or 'campaign' in f:
        os.rename(file, 'intervention_history.csv')
    elif 'churn' in f:
        os.rename(file, 'churn_labels.csv')

# =====================================
# STEP 4: LOAD DATA
# =====================================
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
tickets = pd.read_csv('support_tickets.csv')
web = pd.read_csv('web_events_snapshot.csv')
interventions = pd.read_csv('intervention_history.csv')
churn = pd.read_csv('churn_labels.csv')

print("✅ Data loaded successfully")

# =====================================
# STEP 5: DATE PROCESSING
# =====================================
orders['order_date'] = pd.to_datetime(orders['order_date'])
tickets['ticket_date'] = pd.to_datetime(tickets['ticket_date'])
churn['snapshot_date'] = pd.to_datetime(churn['snapshot_date'])

# =====================================
# STEP 6: FILTER (NO LEAKAGE)
# =====================================
orders = orders.merge(churn[['customer_id','snapshot_date']], on='customer_id')
orders = orders[orders['order_date'] <= orders['snapshot_date']]

tickets = tickets.merge(churn[['customer_id','snapshot_date']], on='customer_id')
tickets = tickets[tickets['ticket_date'] <= tickets['snapshot_date']]

# =====================================
# STEP 7: FEATURE ENGINEERING
# =====================================

# ORDERS
order_agg = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'gross_amount': 'sum'
}).reset_index()

order_agg.columns = ['customer_id','frequency','monetary']

# RECENCY
last_order = orders.groupby('customer_id')['order_date'].max().reset_index()
last_order.columns = ['customer_id','last_order_date']
last_order = last_order.merge(churn[['customer_id','snapshot_date']])
last_order['recency'] = (
    last_order['snapshot_date'] - last_order['last_order_date']
).dt.days

order_agg = order_agg.merge(last_order[['customer_id','recency']], on='customer_id')

# SUPPORT
support_agg = tickets.groupby('customer_id').agg({
    'ticket_id': 'count',
    'sentiment_score': 'mean'
}).reset_index()

support_agg.columns = ['customer_id','ticket_count','avg_sentiment']

# =====================================
# STEP 8: MERGE MODEL DATA
# =====================================
df = churn.merge(order_agg, on='customer_id', how='left')
df = df.merge(support_agg, on='customer_id', how='left')

df.fillna(0, inplace=True)

print("✅ Modeling dataset ready:", df.shape)

# =====================================
# STEP 9: TRAIN MODEL
# =====================================
X = df[['recency','frequency','monetary','ticket_count','avg_sentiment']]
y = df['churn_next_60d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# BASELINE
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# FINAL MODEL
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# =====================================
# STEP 10: EVALUATION
# =====================================
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:,1]

print("\n✅ Model Performance:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

# =====================================
# STEP 11: SAVE MODEL
# =====================================
joblib.dump(rf, 'model.pkl')
print("\n✅ model.pkl saved successfully")